In [1]:
import pandas as pd
import requests

In [36]:
def fetch_weather(city, latitude, longitude):
    url = "https://previous-runs-api.open-meteo.com/v1/forecast"


    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": "2024-03-08",
        "end_date": "2026-04-30",
        "hourly": "temperature_2m_previous_day2,wind_speed_100m_previous_day2",
        "models": "ecmwf_ifs025",
        "timezone": "UTC",
        "wind_speed_unit": "ms"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    weather = pd.DataFrame(response.json()["hourly"])
    
    weather["timestamp"] = pd.to_datetime(weather["time"],
    utc = True)

    weather = weather.rename(columns={
        "temperature_2m_previous_day2": f"temp_{city}",
        "wind_speed_100m_previous_day2": f"wind_{city}"
    })

    return weather[
        ["timestamp", f"temp_{city}", f"wind_{city}"]
        ]   


In [37]:
stockholm = fetch_weather(
    "stockholm", 59.3293, 18.0686
)

gothenburg = fetch_weather(
    "gothenburg", 57.7089, 11.9746
)

orebro = fetch_weather(
    "orebro", 59.2753, 15.2134
)

In [38]:
weather_all = stockholm.merge(
    gothenburg,
    on="timestamp",
    how="inner"
).merge(
    orebro,
    on="timestamp", 
    how="inner"
)

In [39]:
weather_all.head()

,timestamp,temp_stockholm,wind_stockholm,temp_gothenburg,wind_gothenburg,temp_orebro,wind_orebro
0,2024-03-08 00:00:00+00:00,-1.0,5.15,-0.8,3.99,-1.5,5.64
1,2024-03-08 01:00:00+00:00,-1.3,5.21,-1.0,3.79,-1.8,5.64
2,2024-03-08 02:00:00+00:00,-1.7,5.50,-1.0,3.49,-2.1,5.32
3,2024-03-08 03:00:00+00:00,-1.9,5.71,-1.1,3.45,-2.4,5.10
4,2024-03-08 04:00:00+00:00,-2.1,5.92,-1.4,3.86,-2.8,5.05


In [40]:
weather_all.shape

(18816, 7)

In [43]:
weather_all["timestamp"].min(), weather_all["timestamp"].max()

(Timestamp('2024-03-08 00:00:00+0000', tz='UTC'),
 Timestamp('2026-04-30 23:00:00+0000', tz='UTC'))

In [44]:
weather_all.isna().sum()

timestamp          0
temp_stockholm     0
wind_stockholm     0
temp_gothenburg    0
wind_gothenburg    0
temp_orebro        0
wind_orebro        0
dtype: int64

In [45]:
weather_all["timestamp"].duplicated().sum()

np.int64(0)

In [46]:
weather_all["timestamp"].diff().value_counts()

timestamp
0 days 01:00:00    18815
Name: count, dtype: int64

In [47]:
gaps = weather_all[
    weather_all["timestamp"].diff() != pd.Timedelta(hours=1)
]

gaps.head()

,timestamp,temp_stockholm,wind_stockholm,temp_gothenburg,wind_gothenburg,temp_orebro,wind_orebro
0,2024-03-08 00:00:00+00:00,-1.0,5.15,-0.8,3.99,-1.5,5.64


In [48]:
weather_all.shape

(18816, 7)

In [50]:
train_weather = weather_all[
    (weather_all["timestamp"] >= "2024-03-08") &
    (weather_all["timestamp"] < "2025-03-08")
]

validation_weather = weather_all[
    (weather_all["timestamp"] >= "2025-03-08") &
    (weather_all["timestamp"] < "2025-05-01")
]

test_weather = weather_all[
    (weather_all["timestamp"] >= "2025-05-01") &
    (weather_all["timestamp"] < "2026-05-01")
]

print ("Train:", len(train_weather))
print ("Validation:", len(validation_weather))
print ("Test:", len(test_weather))

Train: 8760
Validation: 1296
Test: 8760


In [51]:
weather_all.isna().sum()

timestamp          0
temp_stockholm     0
wind_stockholm     0
temp_gothenburg    0
wind_gothenburg    0
temp_orebro        0
wind_orebro        0
dtype: int64

In [52]:
weather_all["timestamp"].duplicated().sum()

np.int64(0)

In [54]:
weather_all["timestamp"].diff().value_counts().head()

timestamp
0 days 01:00:00    18815
Name: count, dtype: int64

In [55]:
weather_all.to_csv(
    "../data/processed/se3_weather_forecast.csv", 
    index=False
)